In [19]:
# bibliotecas
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [23]:
transform = transforms.Compose([
    transforms.ToTensor(),# imagem pra tensor
    transforms.Normalize((0.5,), (0.5,)) # normaliza
])

train_loader = DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=64, shuffle=True # 64 imagens por batch
)

In [ ]:
# modelo CNN autoencoder
class CNNAE(nn.Module):
    def __init__(self):
        super().__init__()

        # extrai features
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU()
        )

        # reconstrói a imagem expandindo
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Tanh()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x)) # comprime e reconstroi

model = CNNAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001) # otimizador Adam
criterion = nn.MSELoss()

In [ ]:
# treino
for epoch in range(5):
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        recon = model(imgs)
        loss = criterion(recon, imgs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"epoca {epoch+1} loss: {loss.item():.4f}")


epoca 1 loss: 0.0013
epoca 2 loss: 0.0013
epoca 3 loss: 0.0010
epoca 4 loss: 0.0009
epoca 5 loss: 0.0007
